In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os

from catboost import Pool, CatBoostClassifier
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch

## Read Data

In [ ]:
train_data_bank = pd.read_csv("sentirueval/bank_train.csv", engine="c")
train_data_tkk = pd.read_csv("sentirueval/tkk_train.csv", engine="c")

test_data_bank = pd.read_csv("sentirueval/bank_etalon.csv", engine="c")
test_data_tkk = pd.read_csv("sentirueval/tkk_etalon.csv", engine="c")
print(f"Number of rows and columns in the train data set: {train_data_bank.shape}")
print(f"Number of rows and columns in the train data set: {train_data_tkk.shape}")
print(f"Number of rows and columns in the test bank data set: {test_data_bank.shape}")
print(f"Number of rows and columns in the test tkk data set: {test_data_tkk.shape}")
print(train_data_bank.head())
print(train_data_tkk.head())
print(test_data_bank.head())
print(test_data_tkk.head())

## Preprocess Data

In [ ]:
import re

def paragraph_clean(paragraph: str) -> list[str]:
    sentences = paragraph.strip().split(".")
    sentences = '. '.join(' '.join(i.strip().split()) for i in sentences if i)
    return sentences

def clean_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    return text


In [ ]:
train_data_bank_clean = train_data_bank.copy()
train_data_tkk_clean = train_data_tkk.copy()

train_data_bank_list = [i for i in train_data_bank_clean['text']]
train_data_tkk_list = [i for i in train_data_tkk_clean['text']]


test_data_bank_clean = test_data_bank.copy()
test_data_tkk_clean = test_data_tkk.copy() 

test_data_bank_list = [i for i in test_data_bank_clean['text']]
test_data_tkk_list = [i for i in test_data_tkk_clean['text']]


## Getting Embeddings

In [ ]:
model_path = 'ai-forever/FRIDA'
# model_path = 'sergeyzh/BERTA'

In [ ]:
from sentence_transformers import SentenceTransformer

# torch.set_float32_matmul_precision('highest')
device = "cuda:3" if torch.cuda.is_available() else "cpu"
print(device)
model = SentenceTransformer(model_path)
model = model.to(device)


In [ ]:
model = model.to(device)
train_bank_embeddings = model.encode(train_data_bank_list)
train_tkk_embeddings = model.encode(train_data_tkk_list)
test_bank_embeddings = model.encode(test_data_bank_list)
test_tkk_embeddings = model.encode(test_data_tkk_list)
print(train_bank_embeddings.shape)
print(train_tkk_embeddings.shape)
print(test_bank_embeddings.shape)
print(test_tkk_embeddings.shape)


## Catboost preprocess

In [ ]:
train_labels_bank = train_data_bank_clean["label"]
train_features_bank = train_bank_embeddings.copy()

train_labels_tkk = train_data_tkk_clean["label"]
train_features_tkk= train_tkk_embeddings.copy()

train_labels_all = pd.concat([train_labels_bank, train_labels_tkk])
train_features_all = np.vstack((train_features_bank, train_features_tkk))

test_labels_bank = test_data_bank_clean["label"]
test_features_bank = test_bank_embeddings.copy()

test_labels_tkk = test_data_tkk_clean["label"]
test_features_tkk = test_tkk_embeddings.copy()


## Catboost train

In [ ]:
from catboost import Pool, CatBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical


def print_metrics(test_labels, y_pred):
    f1_macro = f1_score(test_labels, y_pred, average='macro')
    f1_micro = f1_score(test_labels, y_pred, average='micro')
    f1_weighted = f1_score(test_labels, y_pred, average='weighted')
    clf_report = classification_report(test_labels, y_pred, digits=4)
    precision = precision_score(test_labels, y_pred, average='macro')
    recall = recall_score(test_labels, y_pred, average='macro')
    print(f"{precision=}, {recall=}, {f1_macro=}, {f1_micro=}, {f1_weighted=}")
    print(clf_report)


def grid_search(train_features, train_labels, scoring, is_grid_search=True):

    cb_model = CatBoostClassifier(task_type="GPU",
                               devices='3',
                               auto_class_weights='Balanced',
                               random_seed=42,
                               verbose=False)
    if is_grid_search:
        param_grid = {
                        'learning_rate': [0.01, 0.1, 0.3],
                        'depth': [1, 3, 5, 7], 
                        'l2_leaf_reg': [0.1, 1, 3], 
                        'min_data_in_leaf': [1, 3]  
                     }
        
        grid_search = GridSearchCV(
                                    estimator=cb_model,
                                    param_grid=param_grid,
                                    scoring=scoring,
                                    cv=3, 
                                    verbose=4,
                                    n_jobs=1 
                                   )
        
        grid_search.fit(train_features, train_labels)
        print("Наилучшие параметры:", grid_search.best_params_)
        best_model = grid_search.best_estimator_
    
    else:
        search_space = {
                    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
                    'depth': Integer(4, 10),
                    'l2_leaf_reg': Real(0.1, 15, prior='log-uniform'),
                    'random_strength': Real(0.01, 10, prior='uniform'),
                    'grow_policy': Categorical(['SymmetricTree', 'Depthwise', 'Lossguide']),
                    'min_data_in_leaf': Integer(1, 10)
                }
    
    
        bayes_search = BayesSearchCV(
            estimator=cb_model,
            search_spaces=search_space,
            n_iter=70, 
            cv=3,
            scoring=scoring,
            random_state=42,
            verbose=2,
            n_jobs=1 
        )
    
    
        bayes_search.fit(train_features, train_labels)
        print("Наилучшие параметры:", bayes_search.best_params_)
        best_model = bayes_search.best_estimator_
    return best_model




In [ ]:
grid_model_tkk = grid_search(train_features_tkk, train_labels_tkk, 'f1_weighted')
y_pred_tkk = grid_model_tkk.predict(test_features_tkk)
print("Telecom metrics")
print_metrics(test_labels_tkk, y_pred_tkk)

grid_model_bank = grid_search(train_features_all, train_labels_all, 'f1_weighted')
y_pred_bank = grid_model_bank.predict(test_features_bank)
print("Bank metrics")
print_metrics(test_labels_bank, y_pred_bank)



In [ ]:
print("Telecom metrics")
print_metrics(test_labels_tkk, y_pred_tkk)
print("Bank metrics")
print_metrics(test_labels_bank, y_pred_bank)